In [2]:
pip install requests beautifulsoup4 pandas


Note: you may need to restart the kernel to use updated packages.


In [30]:
pip install selenium webdriver-manager


Note: you may need to restart the kernel to use updated packages.


In [37]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import time
import pandas as pd
from webdriver_manager.chrome import ChromeDriverManager

# Set up Selenium WebDriver
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--headless")  # Run in headless mode
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

# URL of Cars24 used cars listing
url = "https://www.cars24.com/buy-used-cars-ghaziabad/?sort=bestmatch&serveWarrantyCount=true&storeCityId=132"
driver.get(url)

# Infinite scrolling logic
scroll_pause_time = 2  # Time to wait for new content
max_scrolls = 50  # Maximum scroll attempts to avoid infinite loop
scroll_count = 0

while scroll_count < max_scrolls:
    driver.find_element(By.TAG_NAME, "body").send_keys(Keys.END)  # Scroll to the bottom
    time.sleep(scroll_pause_time)  # Wait for the page to load

    # Check if new cars loaded
    new_car_elements = driver.find_elements(By.CSS_SELECTOR, "a.styles_carCardWrapper__sXLIp")
    
    if len(new_car_elements) >= 405:  # Stop if all cars are loaded
        break

    scroll_count += 1

# Get final page source
soup = BeautifulSoup(driver.page_source, "html.parser")
driver.quit()

# Extract all car listings
car_elements = soup.find_all("a", class_="styles_carCardWrapper__sXLIp")

print(f"Total Cars Found: {len(car_elements)}\n")  # Print total cars scraped

# List to store car data
cars_data = []

# Extract and save car details
for car in car_elements:
    try:
        name = car.find("div", class_="sc-fLVwEd hRljRx").text.strip()
        details_div = car.find("ul", class_="sc-huvEkS gkjlEH")
        details = details_div.find_all("p") if details_div else []
        Kilometers_Travelled = details[0].text.strip() if len(details) > 0 else "N/A"
        Fuel_Type = details[1].text.strip() if len(details) > 1 else "N/A"
        Transmission = details[2].text.strip() if len(details) > 2 else "N/A"
        Ownership = details[3].text.strip() if len(details) > 3 else "N/A"
        Emi = car.find("div", class_="styles_flexItem__2z__J").text.strip()
        Price = car.find("div", class_="styles_priceWrap__VwWBV").text.strip()

        print(name, Kilometers_Travelled, Fuel_Type, Transmission, Ownership, Emi, Price)
        

        # Append car data to list
        cars_data.append([name, Kilometers_Travelled, Fuel_Type, Transmission, Ownership, Emi, Price])

    except AttributeError:
        continue  # Skip if any data is missing

# Convert to Pandas DataFrame
df = pd.DataFrame(cars_data, columns=["Car Name", "Kilometers Travelled", "Fuel Type", "Transmission", "Ownership", "EMI", "Price"])

# Save to Excel file
df.to_excel("/Users/lokeshadada/Downloads/Ghaziabadused_cars.xlsx", index=False)

print("Data saved successfully to 'used_cars.xlsx'")



Total Cars Found: 40

Data saved successfully to 'used_cars.xlsx'
